# 非评分实验 - 检索指标
---

在本实验中，你将围绕 RAG 系统的检索指标进行实践与分析。RAG 模型通过从知识库检索相关文档来提升生成回答质量。你的目标是评估检索组件，计算 precision 和 recall 指标，以及 context precision 和 context recall。

在本实验中，你将学习：
- 如何计算 precision 与 recall 指标
- 如何在信息检索任务中应用这些指标
- 如何基于具体数据集测试语义检索的能力

你将使用 `sentence-transformers` 库将文本转换为嵌入向量，以便高效计算相似度。要计算检索指标，你需要带标注的数据集。

---
<h4 style="color:black; font-weight:bold;">使用目录</h4>

JupyterLab 提供了便捷方式帮助你浏览本作业。目录位于左侧面板中的目录选项卡，如下图所示。

![TOC Location](images/toc.png)


# 目录
- [ 1 - 数据集](#1)
  - [ 1.1 数据预处理与向量化](#1-1)
  - [ 1.2 检索基础函数](#1-2)
- [ 2 - 检索指标](#2)
  - [ 2.1 Precision](#2-1)
  - [ 2.2 Recall](#2-2)
  - [ 2.3 在若干查询上计算指标](#2-3)


## 1 - 介绍
---

检索指标是 RAG 系统中的核心评估工具。要有效衡量性能，需要带标注的数据集，即对特定查询已知正确答案的数据，从而可将其与 RAG 系统生成结果进行比较。在本实验中，你将使用预标注数据集，并聚焦 Precision 与 Recall 指标。

<div style="text-align: center;">
  <img src="images/precision_recall.png" alt="Description" style="width: 70%;">
</div>

In [1]:
import pandas as pd
from sentence_transformers import SentenceTransformer
import numpy as np
import matplotlib.pyplot as plt
import joblib
import os

<a id='1'></a>
### 1.1 数据集

[20 Newsgroups 数据集](https://scikit-learn.org/0.19/datasets/twenty_newsgroups.html) 是经典文本数据集，包含多个主题的文本及类别标注。这里我们使用 `sklearn.datasets` 模块加载该数据集。

In [ ]:
from sklearn.datasets import fetch_20newsgroups

# 加载 20 Newsgroups 数据集
newsgroups_train = fetch_20newsgroups(subset='train', shuffle=True, random_state=42, data_home='./dataset')

# 转为 DataFrame 以便后续处理
df = pd.DataFrame({
    'text': newsgroups_train.data,
    'category': newsgroups_train.target
})

# 显示数据集的基础信息
print(df.head())
print("\nDataset Size:", df.shape)
print("\nNumber of Categories:", len(newsgroups_train.target_names))
print("\nCategories:", newsgroups_train.target_names)

                                                text  category
0  From: lerxst@wam.umd.edu (where's my thing)\nS...         7
1  From: guykuo@carson.u.washington.edu (Guy Kuo)...         4
2  From: twillis@ec.ecn.purdue.edu (Thomas E Will...         4
3  From: jgreen@amber (Joe Green)\nSubject: Re: W...         1
4  From: jcm@head-cfa.harvard.edu (Jonathan McDow...        14

Dataset Size: (11314, 2)

Number of Categories: 20

Categories: ['alt.atheism', 'comp.graphics', 'comp.os.ms-windows.misc', 'comp.sys.ibm.pc.hardware', 'comp.sys.mac.hardware', 'comp.windows.x', 'misc.forsale', 'rec.autos', 'rec.motorcycles', 'rec.sport.baseball', 'rec.sport.hockey', 'sci.crypt', 'sci.electronics', 'sci.med', 'sci.space', 'soc.religion.christian', 'talk.politics.guns', 'talk.politics.mideast', 'talk.politics.misc', 'talk.religion.misc']


In [3]:
print(f"TEXT:\n\t{df['text'][0]}\nCATEGORY:\n\t{newsgroups_train.target_names[df['category'][0]]}")

TEXT:
	From: lerxst@wam.umd.edu (where's my thing)
Subject: WHAT car is this!?
Nntp-Posting-Host: rac3.wam.umd.edu
Organization: University of Maryland, College Park
Lines: 15

 I was wondering if anyone out there could enlighten me on this car I saw
the other day. It was a 2-door sports car, looked to be from the late 60s/
early 70s. It was called a Bricklin. The doors were really small. In addition,
the front bumper was separate from the rest of the body. This is 
all I know. If anyone can tellme a model name, engine specs, years
of production, where this car is made, history, or whatever info you
have on this funky looking car, please e-mail.

Thanks,
- IL
   ---- brought to you by your neighborhood Lerxst ----





CATEGORY:
	rec.autos


<a id='1-1'></a>
### 1.1 数据预处理与向量化

在本节中，你将先对文本做基础清洗，再使用 `sentence-transformers` 预训练模型将文本向量化。这里使用模型 `BAAI/bge-base-en-v1.5` 对查询句子进行编码。为节省时间，数据集文档嵌入已提前计算完成，因此该模型仅用于向量化提示词。

In [ ]:
# 加载预训练的 sentence-transformer 模型
model_name =  "BAAI/bge-base-en-v1.5"
model = SentenceTransformer(os.path.join(os.environ['MODELS'],model_name))

embedding_vectors = joblib.load('embeddings.joblib')

In [5]:
len(embedding_vectors)

11314

<a id='1-2'></a>
### 1.2 检索基础函数

现在我们通过在预计算嵌入上做相似度搜索，来实现一个基础 RAG 检索机制。下面的代码使用余弦相似度为给定查询找到最相关文档。先定义基础函数。


In [ ]:
def preprocess_text(text):
    """
    对文本做预处理：去除首尾空白字符。

    参数：
    text (str): 待预处理文本。

    返回：
    str: 去除首尾空白后的文本。
    """
    # 示例预处理：移除首尾空白
    text = text.strip()
    return text


def cosine_similarity(v1, array_of_vectors):
    """
    计算一个向量与单个向量（1D）或向量数组（2D）之间的余弦相似度。
    输入为 1D 时返回 float；输入为 2D 时返回 float 列表。
    可安全处理 PyTorch 张量（会先移到 CPU）与 NumPy 数组。
    """
    # 处理 v1 为 torch 张量的情况
    if hasattr(v1, "detach"):  # torch tensor
        v1 = v1.detach().cpu().numpy()
    v1 = np.asarray(v1, dtype=np.float32).ravel()

    # 处理 array_of_vectors 为 torch 张量的情况
    if hasattr(array_of_vectors, "detach"):  # torch tensor
        array_of_vectors = array_of_vectors.detach().cpu().numpy()
    A = np.asarray(array_of_vectors, dtype=np.float32)

    if A.ndim == 1:
        A = A.ravel()
        denom = np.linalg.norm(v1) * np.linalg.norm(A)
        return float(0.0 if denom == 0 else np.dot(v1, A) / denom)

    # 2D 情况：对 A 的每一行计算相似度
    A = np.atleast_2d(A)
    v1_norm = np.linalg.norm(v1)
    A_norms = np.linalg.norm(A, axis=1)
    denom = v1_norm * A_norms
    with np.errstate(divide='ignore', invalid='ignore'):
        sims = (A @ v1) / np.where(denom == 0, 1.0, denom)
    sims[denom == 0] = 0.0
    return sims.tolist()


def top_k_greatest_indices(lst, k):
    """
    获取列表中数值最大的前 k 个元素对应索引。

    参数：
    lst (list): 待评估元素列表。
    k (int): 需要返回索引的前 k 个元素数量。

    返回：
    list: 对应前 k 个最大元素的索引列表。
    """
    # 为列表加上索引
    indexed_list = list(enumerate(lst))
    # 按元素值降序排序
    sorted_by_value = sorted(indexed_list, key=lambda x: x[1], reverse=True)
    # 取前 k 个索引
    top_k_indices = [index for index, value in sorted_by_value[:k]]
    return top_k_indices

现在定义检索函数。

In [ ]:
def retrieve_documents(query, embeddings, model, top_k=5):
    """
    使用余弦相似度检索与 query 最相似的 top-k 文档。
    假设：
      - preprocess_text、top_k_greatest_indices、df、newsgroups_train 已在其他位置定义。
      - embeddings 为文档嵌入的可迭代对象（NumPy 数组或 torch 张量）。
      - model.encode 支持 convert_to_tensor 参数（例如 sentence-transformers）。
    """
    
    query_clean = preprocess_text(query)
    query_embedding = model.encode(query_clean, convert_to_tensor=False).astype(np.float32)

    cosine_scores = []
    for x in embeddings:
        # 确保每个嵌入都是 NumPy 数组
        if hasattr(x, "detach"):  # torch tensor
            x = x.detach().cpu().numpy()
        x = np.asarray(x, dtype=np.float32)

        score = cosine_similarity(query_embedding, x)  # 当 x 为 1D 时返回 float
        cosine_scores.append(float(score))

    top_results = top_k_greatest_indices(cosine_scores, k=top_k)

    print(f"Query: {query}")
    for idx in top_results:
        print(f"Document: {df.iloc[idx]['text'][:200]}...")
        print(f"Category: {newsgroups_train.target_names[df.iloc[idx]['category']]}...")
        print("\n\n")

        
# 示例查询
example_query = "space exploration"
retrieve_documents(example_query, embedding_vectors, model, top_k = 2)

Query: space exploration
Document: From: u1452@penelope.sdsc.edu (Jeff Bytof - SIO)
Subject: End of the Space Age?
Organization: San Diego Supercomputer Center @ UCSD
Lines: 16
Distribution: world
NNTP-Posting-Host: penelope.sdsc.edu

...
Category: sci.space...



Document: From: dennisn@ecs.comm.mot.com (Dennis Newkirk)
Subject: Space class for teachers near Chicago
Organization: Motorola
Distribution: usa
Nntp-Posting-Host: 145.1.146.43
Lines: 59

I am posting this for...
Category: sci.space...





<a id='2'></a>
## 2 - 检索指标

---

下面简要介绍检索系统中最常见的指标：Precision@K 与 Recall@K。

<a id='2-1'></a>
### 2.1 Precision@K

Precision@K 用于评估 Top K 检索结果的相关性。其计算方式为：Top K 结果中相关文档数量与 K（总检索文档数）之比。

$$\text{Precision@K} = \frac{\text{Number of Relevant Documents in Top K}}{\text{K}}$$

其中 K 是检索返回的文档数量。

In [ ]:
def precision_at_k(relevant_count, k):
    """
    计算检索系统的 Precision@K。

    Precision@K 表示 Top K 检索结果中相关文档数量
    占总检索文档数 K 的比例。

    参数:
        relevant_count (int): Top K 结果中的相关文档数。
        k (int): 检索返回文档总数（K）。

    返回:
        float: Precision@K 数值；若 k 为 0 则返回 0.0。
    
    异常:
        ValueError: 当输入存在负数时抛出。
    """
    if relevant_count < 0 or k < 0:
        raise ValueError("所有输入值必须为非负数。")
    
    if k == 0:
        return 0.0

    return relevant_count / k

<a id='2-2'></a>
### 2.2 Recall@K

Recall@K 衡量检索系统在 Top K 结果中覆盖全部相关文档的能力。其计算方式为：Top K 结果中相关文档数量与语料中全部相关文档数量之比。

$$\text{Recall@K} = \frac{\text{Number of Relevant Documents in Top K}}{\text{Total Number of Relevant Documents in Corpus}}$$

In [ ]:
def recall_at_k(relevant_count, total_relevant):
    """
    计算检索系统的 Recall@K。

    Recall@K 表示 Top K 检索结果中的相关文档数量
    占整个语料中相关文档总数的比例。

    参数:
        relevant_count (int): Top K 结果中的相关文档数。
        total_relevant (int): 语料中相关文档总数。

    返回:
        float: Recall@K 数值；若 total_relevant 为 0 则返回 0.0。
    
    异常:
        ValueError: 当输入存在负数时抛出。
    """
    if relevant_count < 0 or total_relevant < 0:
        raise ValueError("所有输入值必须为非负数。")

    if total_relevant == 0:
        return 0.0

    return relevant_count / total_relevant

<a id='2-3'></a>
### 2.3 在若干查询上计算指标

现在在一些预定义查询上计算这些指标。

In [ ]:
# 定义更复杂的测试查询及其期望类别
test_queries = [
    {"query": "advancements in space exploration technology", "desired_category": "sci.space"},
    {"query": "real-time rendering techniques in computer graphics", "desired_category": "comp.graphics"},
    {"query": "latest findings in cardiovascular medical research", "desired_category": "sci.med"},
    {"query": "NHL playoffs and team performance statistics", "desired_category": "rec.sport.hockey"},
    {"query": "impacts of cryptography in online security", "desired_category": "sci.crypt"},
    {"query": "the role of electronics in modern computing devices", "desired_category": "sci.electronics"},
    {"query": "motorcycles maintenance tips for enthusiasts", "desired_category": "rec.motorcycles"},
    {"query": "high-performance baseball tactics for championships", "desired_category": "rec.sport.baseball"},
    {"query": "historical influence of politics on society", "desired_category": "talk.politics.misc"},
    {"query": "latest technology trends in the Windows operating system", "desired_category": "comp.os.ms-windows.misc"}
    
]


In [ ]:
def compute_metrics(queries, embeddings, model, top_k=5):
    """
    针对一组查询与文档嵌入数据，计算 Precision@K 与 Recall@K。
    假设：
      - preprocess_text、top_k_greatest_indices、precision_at_k、recall_at_k、df、newsgroups_train 已在其他位置定义。
      - embeddings 为文档嵌入的列表/可迭代对象（NumPy 数组或 torch 张量）。
      - model.encode 支持 convert_to_tensor 参数（例如 sentence-transformers）。
    """

    results = []

    # 先统一将所有嵌入转换为 NumPy
    np_embeddings = []
    for x in embeddings:
        if hasattr(x, "detach"):  # torch tensor
            x = x.detach().cpu().numpy()
        np_embeddings.append(np.asarray(x, dtype=np.float32).ravel())
    E = np.vstack(np_embeddings)  # shape: (N, D)

    for item in queries:
        query = item["query"]
        desired_category = item["desired_category"]

        # 获取 NumPy（非 torch），避免 GPU->NumPy 转换错误
        q_clean = preprocess_text(query)
        q_emb = model.encode(q_clean, convert_to_tensor=False)
        q_emb = np.asarray(q_emb, dtype=np.float32).ravel()

        # 向量化计算相似度
        cosine_scores = cosine_similarity(q_emb, E)  # 长度为 N 的 float 列表

        # Top-K 索引
        top_results = top_k_greatest_indices(cosine_scores, k=top_k)

        # 检索到的类别
        retrieved_categories = [
            newsgroups_train.target_names[df.iloc[idx]["category"]] for idx in top_results
        ]

        # 指标计算
        relevant_in_top_k = sum(1 for cat in retrieved_categories if cat == desired_category)
        total_relevant_in_corpus = sum(
            1 for idx in range(len(df))
            if newsgroups_train.target_names[df.iloc[idx]["category"]] == desired_category
        )

        p = precision_at_k(relevant_in_top_k, top_k)
        r = recall_at_k(relevant_in_top_k, total_relevant_in_corpus)

        results.append({
            "query": query,
            "precision@k": p,
            "recall@k": r,
        })

    return results

In [ ]:
# 运行查询，并在不同 K 下计算指标
k_values = [5, 20, 50]

for k in k_values:
    print(f"\n{'='*80}")
    print(f"Results with K={k}:")
    print('='*80)
    results = compute_metrics(test_queries, embedding_vectors, model, top_k=k)
    
    # 展示结果
    for result in results:
        print(f"Query: {result['query']}")
        print(f"  Precision@{k}: {result['precision@k']:.2f}, Recall@{k}: {result['recall@k']:.2f}")
        print()


Results with K=5:
Query: advancements in space exploration technology
  Precision@5: 1.00, Recall@5: 0.01

Query: real-time rendering techniques in computer graphics
  Precision@5: 1.00, Recall@5: 0.01

Query: latest findings in cardiovascular medical research
  Precision@5: 1.00, Recall@5: 0.01

Query: NHL playoffs and team performance statistics
  Precision@5: 1.00, Recall@5: 0.01

Query: impacts of cryptography in online security
  Precision@5: 1.00, Recall@5: 0.01

Query: the role of electronics in modern computing devices
  Precision@5: 1.00, Recall@5: 0.01

Query: motorcycles maintenance tips for enthusiasts
  Precision@5: 1.00, Recall@5: 0.01

Query: high-performance baseball tactics for championships
  Precision@5: 1.00, Recall@5: 0.01

Query: historical influence of politics on society
  Precision@5: 0.40, Recall@5: 0.00

Query: latest technology trends in the Windows operating system
  Precision@5: 0.80, Recall@5: 0.01


Results with K=20:
Query: advancements in space explor

**结果解读：**

上面的结果清晰展示了当 K 从 5 增加到 20、50 时，检索系统中的 **precision-recall 权衡**：

**Precision@K 趋势（通常随 K 增大而下降）：**

- **当 K=5**：大多数查询 precision 很高（0.80-1.00），10 个查询中有 8 个达到 1.00，说明检索结果几乎都高度相关。
  
- **当 K=20**：部分查询 precision 开始下降：
  - "electronics in computing devices" 下降到 0.80（原为 1.00）
  - "Windows operating system" 下降到 0.65（原为 0.80）
  - "motorcycles maintenance" 下降到 0.95（原为 1.00）
  
- **当 K=50**：随着检索文档增多，precision 进一步下降：
  - "computer graphics" 下降到 0.88（原为 1.00）
  - "electronics in computing devices" 下降到 0.66（原为 0.80）
  - "Windows operating system" 下降到 0.60（原为 0.65）
  - "historical influence of politics" 约在 0.50-0.52（各 K 中最低）

**Recall@K 趋势（随 K 增大而上升）：**

- **当 K=5**：Recall 很低（约 0.01，即 1%），仅检索出全部相关文档中的约 1%。
  
- **当 K=20**：Recall 提升到约 0.03（3%），覆盖了更多相关文档。
  
- **当 K=50**：Recall 提升到 0.05-0.08（5-8%），约为 K=5 时的 8 倍覆盖量。

**关键观察：**

1. **权衡关系非常明显**：K 增大时，召回更多相关文档（Recall 上升），但也会引入更多不相关文档（Precision 下降）。

2. **不同查询难度不同**："historical influence of politics on society" 的 precision 持续最低（0.40-0.52），说明该查询语义更模糊，或类别 "talk.politics.misc" 与相邻类别更难区分。

3. **对 RAG 来说，K=5 到 K=20 往往较合适**：这些取值通常能保持较高 precision（多数检索文档相关）并控制上下文长度。尽管 recall 偏低，但 RAG 目标通常是找出*最相关*文档，而非找全*所有*相关文档。

4. **即使 K=50，Recall 仍相对较低**：这是预期现象。每个类别通常有数百篇文档（500-600），检索 50 篇仅覆盖约 8-10% 的相关文档。若要显著提高 recall，需要 K 达到数百，这会明显损害 precision，且在 RAG 场景下不够实用。

恭喜你完成这个非评分实验！继续加油！